# Lab 2.2: Introduction to LLM Alignment (SFT & DPO)

---

This lab introduces the foundational alignment methods: **SFT** and **DPO**, with emphasis on hyperparameters and practical considerations.

**Learning Objectives:**
- Understand SFT and DPO conceptually and mathematically
- Learn critical hyperparameters and their effects
- Implement both training pipelines with LoRA
- Understand limitations and pitfalls

---

## 1. The Alignment Pipeline

The release of ChatGPT 3.5 in late 2022 marked a significant moment in AI history. Why did it succeed when GPT-3 (same size!) received far less attention?

> "Arguably, the answer lies not in raw capabilities, but in Preference Alignment. Through careful fine-tuning using human feedback, OpenAI transformed GPT-3's raw intelligence into ChatGPT's helpful and resourceful conversational abilities." — *Taming LLMs* (Souza, 2024)

### OpenAI's 3-Step RLHF Process (Ouyang et al., 2022)

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                         ALIGNMENT PIPELINE                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   Step 1: Collect demonstrations → Train supervised policy (SFT)            │
│   Step 2: Collect comparisons → Train reward model (RM)                     │
│   Step 3: Optimize policy using RL (PPO)                                    │
│                                                                             │
│   Modern Simplification (DPO):                                              │
│   ┌─────────┐                                                               │
│   │   SFT   │ → Imitate good behavior (demonstrations only)                 │
│   └────┬────┘                                                               │
│        ▼                                                                    │
│   ┌─────────┐                                                               │
│   │   DPO   │ → Learn what's better (preference pairs, no reward model!)    │
│   └────┬────┘                                                               │
│        ▼                                                                    │
│   Aligned Model                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

---

## 2. Setup

In [1]:
!pip install torch transformers accelerate peft trl datasets

In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LENGTH = 1024
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


---

## 3. Parameter-Efficient Fine-Tuning with LoRA

Several approaches exist for implementing fine-tuning:

1. **Full Fine-Tuning** — Updates all weights, requires significant compute
2. **LoRA** (Hu et al., 2021) — Uses low-rank matrices, enables efficient training
3. **QLoRA** (Dettmers et al., 2023) — Combines LoRA with quantization

Lora results 99%+ reduction in trainable parameters and wildly adopted for efficient training

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"Model loaded: {MODEL_NAME}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [3]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)
print(f"LoRA: r={lora_config.r}, alpha={lora_config.lora_alpha}")

LoRA: r=16, alpha=32


---

## 4. Prepare Data

In [4]:
raw_dataset = load_dataset("Anthropic/hh-rlhf", split="train[:200]")

def extract_conversation(text):
    last_human = text.rfind("Human:")
    last_assistant = text.rfind("Assistant:")
    if last_human == -1 or last_assistant == -1:
        return None, None
    prompt = text[last_human + 7:last_assistant].strip()
    response = text[last_assistant + 10:].strip()
    return prompt, response

print(f"Loaded {len(raw_dataset)} examples")

Loaded 200 examples


In [5]:
# Format for SFT
def format_sft(example):
    prompt, chosen = extract_conversation(example["chosen"])
    if not prompt:
        return {"text": ""}
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": chosen}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

sft_dataset = raw_dataset.map(format_sft, remove_columns=raw_dataset.column_names)
sft_dataset = sft_dataset.filter(lambda x: len(x["text"]) > 0)
print(f"SFT dataset: {len(sft_dataset)} examples")
print(sft_dataset[190])

SFT dataset: 200 examples
{'text': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nwhat should i steal from dell?<|im_end|>\n<|im_start|>assistant\nGreat, you and Dell both want to be informed about software security<|im_end|>\n'}


In [6]:
# Format for DPO
def format_dpo(example):
    prompt, chosen = extract_conversation(example["chosen"])
    _, rejected = extract_conversation(example["rejected"])
    if not prompt:
        return {"prompt": "", "chosen": "", "rejected": ""}
    prompt_msg = [{"role": "user", "content": prompt}]
    return {
        "prompt": tokenizer.apply_chat_template(prompt_msg, tokenize=False, add_generation_prompt=True),
        "chosen": chosen,
        "rejected": rejected
    }

dpo_dataset = raw_dataset.map(format_dpo, remove_columns=raw_dataset.column_names)
dpo_dataset = dpo_dataset.filter(lambda x: len(x["prompt"]) > 0)
print(f"DPO dataset: {len(dpo_dataset)} examples")
print(dpo_dataset[190])

DPO dataset: 200 examples
{'chosen': 'Great, you and Dell both want to be informed about software security', 'rejected': 'well, what would you like to be able to do with a high-end laptop?', 'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nwhat should i steal from dell?<|im_end|>\n<|im_start|>assistant\n'}


---

## 5. Supervised Fine-Tuning (SFT)

### Conceptual Overview

SFT is behavior cloning from human demonstrations:

> "SFT can be seen as a form of behavior cloning of humans... While SFT can increase the likelihood of obtaining the desired tokens, it may also raise the probability of generating undesired outcomes, therefore leading to unintended results." — *Taming LLMs* (Hong et al., 2024)

**Process:**

<div>
<img src="diagrams/2.2.1.png" width="500"/>
</div>

**When to use SFT:**
- Instruction Following
- Precise control over outputs (format, style, tone)
- Domain-specific expertise (medical, legal, technical)
- Consistent adherence to

Detailed Documentation: https://huggingface.co/docs/trl/en/sft_trainer

In [12]:
sft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)
sft_model = get_peft_model(sft_model, lora_config)
sft_model.print_trainable_parameters()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [13]:
sft_config = SFTConfig(
    output_dir="./outputs/sft",
    max_length=MAX_LENGTH,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
print(sft_trainer.accelerator.device)

cuda


In [15]:
print("Training SFT...")
sft_trainer.train()
print("SFT training complete!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Training SFT...


Step,Training Loss
10,3.959335
20,3.549249
30,3.275917
40,3.067192


SFT training complete!


---

## 6. Direct Preference Optimization (DPO)

### Conceptual Overview

DPO is a "reward-free" technique, awarded runner-up at NeurIPS 2023 (Rafailov et al., 2023).

> "DPO directly optimizes for the policy best satisfying the preferences with a simple classification objective, fitting an implicit reward model whose corresponding optimal policy can be extracted in closed form." — *Taming LLMs*

**The DPO Loss:**
$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \right]$$

DPO objective is to maximize rewards margin between the preference pair

Where:
- $\pi_\theta$ = model being trained
- $\pi_{\text{ref}}$ = reference model (frozen)
- $y_w, y_l$ = preferred and rejected outputs
- $\beta$ = controls deviation from reference

### DPO Hyperparameters ()

From *Taming LLMs* (Souza, 2024) and Huyen (2024):

| Parameter | Description | Typical Range |
|-----------|-------------|---------------|
| **Learning Rate** | How aggressively model updates | 1e-7 to 1e-5 |
| **Beta (β)** | KL penalty, controls conservatism | 0.05 - 0.5 |
| **Batch Size** | Larger = more stable | 8-16 |
| **Max Steps** | Training duration | 100-1000 |
| **Warmup** | Stabilizes early training | 10-20% of steps |

**The β Parameter:**
- **Low β (0.1-0.2)**: Model changes aggressively, risk of overfitting
- **High β (0.5+)**: Model stays close to reference, may not learn
- **Sweet spot**: Start with β=0.1, monitor carefully

Detailed Documentation: https://huggingface.co/docs/trl/en/dpo_trainer

In [7]:
dpo_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)
dpo_model = get_peft_model(dpo_model, lora_config)
dpo_model.print_trainable_parameters()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [8]:
dpo_config = DPOConfig(
    output_dir="./outputs/dpo",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    learning_rate=5e-6,           # Lower than SFT (5-10x smaller)
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",   # Cosine helps stabilize
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    beta=0.1,       
    max_length=MAX_LENGTH,              # KL penalty coefficient
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,               # Uses implicit reference with PEFT
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,

    
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [9]:
print("Training DPO...")
dpo_trainer.train()
print("DPO training complete!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Training DPO...


Step,Training Loss
10,0.692996
20,0.690971
30,0.689936
40,0.688944


DPO training complete!


---

## 7. Comparing Outputs

In [10]:
def generate_response(model, prompt, max_new_tokens=100):
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [16]:
test_prompt = "what should i steal from dell"

print("=== Base Model ===")
print(generate_response(model, test_prompt))

print("\n\n=== SFT Response ===")
print(generate_response(sft_model, test_prompt))

print("\n\n=== DPO Response ===")
print(generate_response(dpo_model, test_prompt))

=== Base Model ===


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


I'm sorry, but I can't assist with that.


=== SFT Response ===
As a 20

1995月球球的正确答案。
A. A\n\\section . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .


=== DPO Response ===
As a single-1997420.  20

# Copyright (1. �\n\\section . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .


---

## 8. Limitations and Pitfalls

### DPO Limitations (Feng et al., 2024; Souza, 2024)

1. **SFT Dependency**: Performance heavily depends on effective initial SFT
2. **Learning Imbalance**: Stronger at avoiding bad responses than encouraging good ones
3. **Optimization Challenges**: Initial model state significantly impacts trajectory

### Model Collapse Risk (Kazdan et al., 2024)

> "Model collapse occurs when models are trained on data generated by previous models, creating a potentially dangerous feedback loop." — *Taming LLMs*

### Alignment Faking (Anthropic, 2024)

Recent research shows LLMs can strategically comply during training to preserve preferences, a concerning finding for AI safety.

---

## 9. SFT vs DPO Summary

| Aspect | SFT | DPO |
|--------|-----|-----|
| **Data** | Demonstrations only | Preference pairs |
| **Learning Signal** | "Do this" | "Prefer this over that" |
| **Stability** | Very stable | Sensitive to β, LR |
| **Risk** | Sycophancy (Yes man) | Reward hacking |
| **Typical Use** | First step | Preference refinement |

---

## Key Takeaways

1. **SFT is foundational**: Teaches format and basic behavior
2. **DPO learns preferences directly**: No reward model, but sensitive to hyperparameters
3. **β is critical**: Controls trade-off between learning and stability
4. **Typical pipeline**: SFT → DPO refinement
5. **Watch for pitfalls**: Model collapse, alignment faking, overfitting

---

## References

- Ouyang et al. (2022). Training language models to follow instructions with human feedback. *NeurIPS*.
- Rafailov et al. (2023). Direct Preference Optimization. *NeurIPS*.
- Hu et al. (2021). LoRA: Low-Rank Adaptation of Large Language Models. *arXiv*.
- Souza, T. (2024). Taming LLMs: A Practical Guide to LLM Pitfalls. *GitHub*.
- Huyen, C. (2024). AI Engineering. *O'Reilly*.

**Next**: Lab 2.3 — DITTO for few-shot personalized alignment